# $Bx$ component as Ito's process 
This notebook is devoted to analysis of connection between $a(t)$ and $b(t)$ 
from Ito equation: 
$$dX = a(t)dt + b(t)dW, \quad \text{where } W \text { is a normal Wiener process}$$

In [ ]:
# Import modules
import numpy as np
import plotly.graph_objects as go
import plotly.express as ple
import plotly.subplots as sp
from magfield.visual.approximation import harmonic_approximation
from magfield.visual.plot import long_plot
from plotly.offline import init_notebook_mode
from IPython.display import display, HTML
import pickle
from tqdm.notebook import tqdm


# Allows you to use modified modules without rebooting the kernel
%load_ext autoreload
%autoreload 2
# Enable latex on plotly figures
init_notebook_mode()
display(
    HTML(
        '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
    )
)

## Calculate $a(t), \space b(t)$

In [ ]:
# Load up gaussian mixture model for component Bx
series_name = "By"
# with open(f"data/d{series_name}_4300_4.pkl", "rb") as f:
with open(f"data/full_d{series_name}_2150_3.pkl", "rb") as f:
    gmm = pickle.load(f)

Parameters of Ito's equation are calculated in the next way:

$$
a(t) = \sum_{k=1}^{K}{p_k a_k}, \quad 
b(t) = \sum_{k=1}^{K}{p_k b_k},
$$
where $K$ is a number of mixture components

In [ ]:
from magfield.em.auxiliary import smooth

p = gmm["weights"]
a = gmm["means"]
b = gmm["variances"]
K = gmm["num_comp"]
nc = gmm["num_comp"]

smoothcoef = True
# smoothcoef = False

coef_a = np.sum(p * a, axis=0)
coef_b = np.sum(p * b, axis=0)

if smoothcoef:
    # smooth_N = int(0.05 * len(coef_a))
    smooth_N = 60 * 12
    coef_a = smooth(coef_a, smooth_N)
    coef_b = smooth(coef_b, smooth_N)

In [ ]:
# Create subplots
fig = sp.make_subplots(
    rows=2,
    cols=1,
    subplot_titles=(
        rf"Ito's coefficient $a(t) = \sum_{{k=1}}^{K}{{p_k a_k}}$",
        rf"Ito's coefficient $b(t) = \sum_{{k=1}}^{K}{{p_k b_k}}$",
    ),
)

# Add traces for the first plot (coef_a)
fig.add_trace(go.Scatter(y=coef_a, mode="lines", name="coef_a"), row=1, col=1)

# Add traces for the second plot (coef_b)
fig.add_trace(go.Scatter(y=coef_b, mode="lines", name="coef_b"), row=2, col=1)

# Update layout for better presentation (optional)
smooth_title = ""
smooth_file = ""
if smoothcoef:
    smooth_title = f" smoothed by {smooth_N} minutes."
    smooth_file = f"_smooth_{smooth_N}"
fig.update_layout(
    height=600,
    width=1800,
    title_text=f"Ito's Coefficients for d{series_name}" + smooth_title,
    showlegend=False,
)

# Show the plot
fig.show()
fig.write_image(f"/tmp/d{series_name}_Ito_coeffs{smooth_file}.png")

# Correlation plots

In [ ]:
correlation_window = (gmm["window"]["size"], gmm["window"]["step"])

#### Correlation $a(t)$, $b(t)=\sum_{j=1}^Kp_jb_j$

In [ ]:
correlation = []
for i in tqdm(range(0, len(coef_a) - correlation_window[0], correlation_window[1])):
    sub_a = coef_a[i : correlation_window[0] + i]
    sub_b = coef_b[i : correlation_window[0] + i]
    correlation.append(np.corrcoef(sub_a, sub_b)[0, 1])

In [ ]:
title = rf"""$\text{{Correlation between }} a(t), b(t)=
    \sum_{{j=1}}^{gmm["num_comp"]}p_jb_j \text{{ on window size }}
    {correlation_window[0]} \text{{ minutes}}$
    """
cp = long_plot(correlation, title, dates=gmm["dates"], n=12)
cp.show()
cp.write_image(f"/tmp/d{series_name}_corr{nc}_{correlation_window[0]}{smooth_file}.png")

In [ ]:
# Create a histogram plot for a correlation of coef_a and coef_b using plolty express

percent = np.count_nonzero(np.array([abs(x) for x in correlation]) > 0.5) / len(
    correlation
)
fig = ple.histogram(
    x=correlation,
    nbins=10,
    labels={"x": "Correlation", "y": "percents"},
    title=f"Percent of values higher than 0.5 for correlation a(t), b(t) of d{series_name} - {round(percent,3)}",
    histnorm="percent",
)
fig.show()
fig.write_image(
    f"/tmp/d{series_name}_histogram_corr{nc}_{correlation_window[0]}{smooth_file}.png"
)

#### Correlation $a(t)$, $b^2(t)$

In [ ]:
# correlation2 = []
# for i in tqdm(range(0, len(coef_a) - correlation_window[0], correlation_window[1])):
#     sub_a = coef_a[i : correlation_window[0] + i]
#     sub_b = (coef_b**2)[i : correlation_window[0] + i]
#     correlation2.append(np.corrcoef(sub_a, sub_b)[0, 1])

In [ ]:
# title = (
#     rf"""$\text{{Correlation between }} a(t), b^2(t)=
#     \left(\sum_{{j=1}}^{gmm["num_comp"]}p_jb_j\right)^2 \text{{ on window size }}
#     {correlation_window[0]} \text{{ minutes}}$""")
# cp = long_plot(correlation2, title)
# cp.show()
# nc = gmm["num_comp"]
# cp.write_image(f"/tmp/dBx_corr_sqr_{nc}_{correlation_window[0]}.png")

# Analysis of $a(t)$ and $b(t)$ relation

## Trigonometric approximation

### Static harmonics for all data

In [ ]:
hn = 8
fig, resids, *_ = harmonic_approximation(
    data=correlation,
    time=gmm["dates"][: len(correlation)],
    harmonics_num=hn,
    title=f"d{series_name} - correlation a(t), b(t)",
)
del _
fig.show()
fig.write_image(f"/tmp/d{series_name}_harmonic_approx_{hn}_{correlation_window[0]}.png")

### Dynamic harmonics

In [ ]:
from magfield.frame import Frame

frame1 = Frame(
    size=gmm["window"]["size"],
    step=3 * 60 * gmm["window"]["step"],
)

signal = correlation[:200_000]
anim_param = dict(
    data=signal,
    duration=10,
    sample_rate=1 / 60,  # one count per 1 minute
    output_path=f"{series_name}_{frame1.size}_animation.gif",
    freq_lim=10,
)
frame1.spectrum_gif(**anim_param)

In [ ]:
# if __name__ == "__main__":
#     fs = 100  # Частота дискретизации
#     t = np.arange(0, 25, 1/fs)
#     signal = np.sin(2 * np.pi * 5 * t)  + 0.5 * np.random.randn(len(t))
#     signal += np.sin(2 * np.pi * 20 * t) + 0.5 * np.random.randn(len(t))

#     create_spectrum_gif(
#         signal=signal,
#         window_size=100,
#         step=50,
#         duration=3,
#         sample_rate=fs,
#         output_path="sa.gif"
#     )

Global plan is to examine periodic behavior in correlation between a(t) and b(t)
of Ito Process. As I understand, it's impossible to approximate yearly correlation
with small number of harmonics.
Therefore I can define shifting window and approximate data on it by small number
of harmonics. Then, shifting the frame, sinusoids parameters would change. 
Meaning they are dynamic in time.

Then I can 

In [ ]:
harmonics_num = 4
window = {"width": gmm["window"]["size"], "step": gmm["window"]["step"]}
data = correlation
for i in tqdm(range(0, len(data) - window["width"], window["step"])):
    d = coef_a[i : correlation_window[0] + i]

## Dynamic autocorrelation regression

In [ ]:
def autocorrelation(data):
    """
    Calculate the autocorrelation of a 1D numpy array.

    Parameters:
    data (numpy.ndarray): 1D array of data.

    Returns:
    numpy.ndarray: Autocorrelation values.
    """
    # Ensure the data is a 1D numpy array
    data = np.asarray(data)
    n = len(data)
    mean = np.mean(data)
    var = np.var(data)
    # Autocorrelation array
    autocorr = np.correlate(data - mean, data - mean, mode="full")[n - 1 :] / (var * n)
    return autocorr

In [ ]:
import statsmodels.api as sm

acorr = sm.tsa.acf(correlation, nlags=10_000)

## Fourier transform for correlation plots

In [ ]:
import plotly.graph_objects as go

import pandas as pd

# Load data
df = pd.DataFrame(
    dict(Date=gmm["dates"][2 * correlation_window[0] :], Values=correlation)
)
# Create figure
fig = go.Figure()

fig.add_trace(go.Scatter(x=list(df.Date), y=list(df.Values)))

# Set title
fig.update_layout(title_text="Time series with range slider and selectors")

# Add range slider
fig.update_layout(
    xaxis=dict(
        rangeselector=dict(
            buttons=list(
                [
                    dict(count=1, label="1m", step="month", stepmode="backward"),
                    dict(count=52, label="52d", step="day", stepmode="backward"),
                    dict(count=3, label="3m", step="month", stepmode="backward"),
                    dict(count=6, label="6m", step="month", stepmode="backward"),
                    dict(step="all"),
                ]
            )
        ),
        rangeslider=dict(visible=True),
        type="date",
    )
)

fig.show()

In [ ]:
from scipy.fft import fft, fftfreq
import numpy as np

# Number of sample points
shift = 10_000
window_size = 1000
N = len(correlation[shift : shift + window_size])
# sample spacing
T = 1.0 / 60.0  # one sample per minute
# x = np.linspace(0.0, N*T, N, endpoint=False)
# y = np.sin(50.0 * 2.0*np.pi*x) + 0.5*np.sin(80.0 * 2.0*np.pi*x)
y = correlation[shift : shift + window_size]
yf = fft(y)
xf = fftfreq(N, T)[: N // 2][:3000]

amp = 2.0 / (N * np.pi * 2.0) * np.abs(yf[0 : N // 2][:3000])
ple.line(x=xf, y=amp, title="Without window function").show()

y = correlation[shift : shift + window_size] * np.hamming(N)
yf = fft(y)
amp = 2.0 / (N * np.pi * 2.0) * np.abs(yf[0 : N // 2][:3000])
ple.line(x=xf, y=amp, title="With Hamming window function").show()

In [ ]:
import scipy.signal as sps

sps.find_peaks(amp, threshold=0.002)